# TP 1 : Ingénierie des Prompts pour les Systèmes Multi-Agents

## Partie 1 : La Tokenisation avec Tiktoken

In [ ]:
import tiktoken

In [ ]:
encoding = tiktoken.encoding_for_model("gpt-4o")
print(encoding.name)

In [ ]:
system_message ="""
Perform Sentiment analysis of the review presented in the user message.
The result should be positive or negative. Do not justify your response
"""
tokens = encoding.encode(system_message)

In [ ]:

print(tokens)

In [ ]:

for token in tokens:
 print(encoding.decode_single_token_bytes(token=token),end="")


In [ ]:
print(len(tokens))

In [ ]:
def num_tokens_from_string(string: str, encoding_name: str = "o200k_base") -> int:
    """Retourne le nombre de tokens dans une chaîne de caractères."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Test de la fonction avec une phrase de votre choix
ma_phrase = "tiktoken is great!"
resultat = num_tokens_from_string(ma_phrase)
print(f"Nombre de tokens pour '{ma_phrase}' : {resultat}")

## Partie 2 : Utilisation d'Ollama avec LangChain

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:3b")
response = llm.invoke([ {"role": "system", "content": "You are a helpful assistant. The output should be in Markdown"},
    {"role": "user", "content": "C'est quoi un Agent AI"}
])


In [ ]:
from IPython.display import display, Markdown                       
print(display(Markdown(response.content)))


In [ ]:
from dotenv.ipython import load_dotenv
load_dotenv(override=True)


In [ ]:
from langchain_groq import ChatGroq
llm2 = ChatGroq(model="openai/gpt-oss-120b")

from langchain.messages import SystemMessage, HumanMessage
resp2 = llm2.invoke([
    SystemMessage("You are a helpful assistant. The output should be in Markdown"),
    HumanMessage("C'est quoi un Agent AI")
])

print(display(Markdown(resp2.content)))



## Partie 3 : Comment rédiger des instructions pour les LLMs OpenAI

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
from langchain_openai import ChatOpenAI

llm3 = ChatOpenAI(
    model="gpt-5.2")



resp = llm3.invoke([
    {"role": "system", "content": system_message},
    {"role": "user", "content": "L'écran est très bon, mais je n'ai pas aimé la souris. le clavier Ma fih Maytchaf"}
])


In [ ]:
from IPython.display import display, Markdown
print(display(Markdown(response.content)))

In [ ]:
system_message = """
Effectuez une analyse de sentiments basée sur les aspects des avis concernant les ordinateurs portables présentés en entrée.
Chaque avis peut comporter un ou plusieurs des aspects suivants : screen, keybloard et pad.
Pour chaque avis présenté en entrée :
- Identifiez la présence d'au moins un des trois aspects (screen, keybloard, pad).
- Attribuez une polarité de sentiment (positive, negative ou neutral) à chaque aspect. Organisez votre réponse dans un objet JSON avec les en-têtes suivants :
  - category:[liste des aspects]
  - polarity:[liste des polarités correspondantes pour chaque aspect]
Si l'un des aspects n'est pas présent dans l'avis de l'utilisateur, tu supposes que la polarité est neutre 
"""


In [ ]:
llm = ChatOpenAI(
    model="gpt-5.2",
    temperature=0,
     model_kwargs={
        "response_format": {"type": "json_object"}
    } 
)

resp = llm.invoke([
    {"role": "system", "content": system_message},
    {"role": "user", "content": "L'écran est très bon, mais je n'ai pas aimé la souris. le clavier Ma fih Maytchaf"}
])


In [ ]:
resp = llm.invoke([
    {"role": "system", "content": system_message},
    {"role": "user", "content": "L'écran est très bon, mais je n'ai pas aimé la souris. le clavier Ma fih Maytchaf"}
])

In [ ]:
print(resp.content)

## 4. LLM Multi-Modal : Génération d'image (Text-to-Image)

In [ ]:
import base64
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage  # <-- L'importation correcte selon les normes
from langchain_openai import ChatOpenAI


load_dotenv(override=True)

llm = ChatOpenAI(model="gpt-5.2")
llm_with_tools = llm.bind_tools([{"type": "image_generation", "quality": "high"}])

response = llm_with_tools.invoke(
    [HumanMessage(content="Je veux une photo d'un chat qui code du Java.")]
)

image_block = response.content_blocks[0]
image_bytes = base64.b64decode(image_block["base64"])

with open("generated_cat_java.png", "wb") as output_file:
    output_file.write(image_bytes)

print("Image generee: generated_cat_java.png")

## 5. LLM Multi-Modal : Description d'image (Image-to-Text)

In [ ]:
import base64
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

load_dotenv(override=True)

image_path = "rag.png"
img = encode_image(image_path)

llm = ChatOpenAI(model="gpt-5.2")

response = llm.invoke(
    [
        HumanMessage(
            content=[
                {"type": "text", "text": "Qu'est-ce que tu vois dans cette image ?"},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}},
            ]
        )
    ]
)

print(response.content)